# The Obama backdrop: leaves and flowers

A notebook just for experimenting with the backdrop behind the cube model.

The direction comes from Kehinde Wiley's official portrait of Barack Obama (2018): a dense wall of saturated green leaves, with chrysanthemums for Chicago, jasmine for Hawaii, and African blue lilies for Kenya. We take only that direction from it. His painting is never fed to a model or stored here.

1. **Leaf wall.** Text-to-image, no flowers, compared across two models: SDXL, and FLUX.1-schnell, which follows prompts better. FLUX needs an L4 or A100 with high RAM; on a T4, skip it.
2. **Flowers, placed exactly.** Real flower photos with open licenses, cut out (`obama_kit/flowers/`), are pasted where `layout.json` says. Only a band of the backdrop ever shows on the cube (`obama_kit/backdrop_seen.png`), so every flower goes in that band. The species are right by construction.
3. **Blend.** A light SDXL img2img pass paints flowers and leaves together.

The result is `backdrop.png`. The cube pipeline cuts it into triangles and dots afterwards.

**Flower photos** (from Wikimedia Commons; cut out and resized):
- Chrysanthemums: "Chrysanthemum × Hortorum Multiflora" by Agbarto, CC0. https://commons.wikimedia.org/wiki/File:2026.10.17_Chrysanthemum_%C3%97_Hortorum_Multiflora.jpg
- Jasmine: "Gelsomino - Jasminum polyanthum - Flowers and buds" by Anna.Massini, CC BY 4.0. https://commons.wikimedia.org/wiki/File:Gelsomino_-_Jasminum_polyanthum_-_Flowers_and_buds.jpg
- African blue lily: "Agapanthus Blue Giant" by Pamela Carls, CC BY 2.0. https://commons.wikimedia.org/wiki/File:Agapanthus_Blue_Giant.jpg

In [ ]:
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 sentencepiece protobuf

In [ ]:
import os, gc, json, urllib.request, torch
from PIL import Image, ImageFilter
from google.colab import files
BRANCH = "cubism/obama"   # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_kit/"
os.makedirs("kit/flowers", exist_ok=True)
os.makedirs("walls", exist_ok=True)
urllib.request.urlretrieve(BASE + "flowers/layout.json", "kit/flowers/layout.json")
urllib.request.urlretrieve(BASE + "backdrop_seen.png", "kit/backdrop_seen.png")
LAYOUT = json.load(open("kit/flowers/layout.json"))
for f in sorted({s["flower"] for s in LAYOUT}):
    urllib.request.urlretrieve(BASE + "flowers/" + f, "kit/flowers/" + f)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU {GPU_GB:.0f} GB;", len(LAYOUT), "flowers to place")

def free():
    for name in ("sdxl", "flux", "blend"):
        if name in globals():
            del globals()[name]
    gc.collect(); torch.cuda.empty_cache()

## 1. The leaf wall

In [ ]:
WALL_PROMPT = ("a dense wall of overlapping green leaves filling the entire frame edge to edge, seen flat from the front "
               "like a hedge, crisp individually painted leaves in many shades of saturated emerald, jade and leaf green, "
               "deep shadowy gaps between the leaves, decorative, richly detailed, vivid, painterly realism")
WALL_NEGATIVE = "flowers, blossoms, petals, sky, horizon, ground, person, face, text, signature, watermark, frame, blurry, muted, dull"
WALL_SEEDS = [1, 2]

In [ ]:
# SDXL
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
free()
sdxl = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16",
    vae=AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16))
sdxl.enable_model_cpu_offload() if GPU_GB < 20 else sdxl.to("cuda")
for s in WALL_SEEDS:
    sdxl(prompt=WALL_PROMPT, negative_prompt=WALL_NEGATIVE, width=1024, height=1024, num_inference_steps=35,
         guidance_scale=6.5, generator=torch.Generator("cuda").manual_seed(s)).images[0].save(f"walls/sdxl_{s}.png")
free()

In [ ]:
# FLUX.1-schnell (L4 or A100; skip on a T4). It has no negative prompt, so the prompt says "no flowers" itself.
from diffusers import FluxPipeline
free()
flux = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16)
flux.enable_model_cpu_offload() if GPU_GB >= 30 else flux.enable_sequential_cpu_offload()
for s in WALL_SEEDS:
    flux(prompt=WALL_PROMPT + ". Only leaves, no flowers at all.", width=1024, height=1024, num_inference_steps=4,
         guidance_scale=0.0, max_sequence_length=256,
         generator=torch.Generator("cpu").manual_seed(s)).images[0].save(f"walls/flux_{s}.png")
free()

In [ ]:
walls = sorted(os.listdir("walls"))
grid = Image.new("RGB", (330 * len(walls), 330), "white")
for i, w in enumerate(walls):
    grid.paste(Image.open(f"walls/{w}").resize((320, 320)), (330 * i, 0))
print(walls)
grid

In [ ]:
WALL = "walls/sdxl_1.png"   # pick one from the list above

## 2. Flowers, placed exactly
Each flower in `layout.json` has a center, a size (pixels on a 1024 backdrop) and a turn. The darkened preview shows the band of backdrop that ever appears on the cube.

In [ ]:
def place(wall, layout, cutouts="kit/flowers", shadow=0.35):
    W = wall.size[0]; k = W / 1024
    out = wall.convert("RGB").copy()
    mask = Image.new("L", wall.size, 0)
    for spot in layout:
        im = Image.open(f"{cutouts}/{spot['flower']}").convert("RGBA")
        if spot.get("mirror"):
            im = im.transpose(Image.FLIP_LEFT_RIGHT)
        s = spot["size"] * k / max(im.size)
        im = im.resize((max(1, int(im.width * s)), max(1, int(im.height * s))), Image.LANCZOS)
        im = im.rotate(spot.get("turn", 0), expand=True, resample=Image.BICUBIC)
        x, y = int(spot["x"] * k - im.width / 2), int(spot["y"] * k - im.height / 2)
        a = im.getchannel("A")
        sh = a.filter(ImageFilter.GaussianBlur(10 * k)).point(lambda v: int(v * shadow))
        out.paste(Image.new("RGB", im.size, (10, 25, 15)), (x + int(6 * k), y + int(8 * k)), sh)
        out.paste(im.convert("RGB"), (x, y), a)
        mask.paste(255, (x, y), a)
    return out, mask

collage, flower_mask = place(Image.open(WALL).convert("RGB").resize((1024, 1024)), LAYOUT)
seen = Image.open("kit/backdrop_seen.png").convert("L").resize((1024, 1024))
preview = Image.composite(collage, Image.blend(collage, Image.new("RGB", collage.size, "black"), 0.65), seen)
collage.save("collage.png")
preview

## 3. Blend
One light SDXL img2img pass, so the photos and the leaves look painted together. **BLEND_STRENGTH** around 0.25–0.35 keeps each species clear; higher repaints more and starts to change the flowers. **KEEP** puts back that share of the original flowers afterwards, if the pass blurred them.

In [ ]:
BLEND_PROMPT = ("a lush painted wall of saturated green leaves with orange-red chrysanthemums, clusters of small white "
                "jasmine flowers and blue African lilies (agapanthus), vivid, decorative, crisp, painterly realism")
BLEND_NEGATIVE = "text, signature, watermark, frame, blurry, muted, dull, person, face"
BLEND_STRENGTH, KEEP, BLEND_SEED = 0.3, 0.3, 4

from diffusers import StableDiffusionXLImg2ImgPipeline, AutoencoderKL
free()
blend = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16",
    vae=AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16))
blend.enable_model_cpu_offload() if GPU_GB < 20 else blend.to("cuda")
painted = blend(prompt=BLEND_PROMPT, negative_prompt=BLEND_NEGATIVE, image=collage, strength=BLEND_STRENGTH,
                num_inference_steps=40, guidance_scale=6.0,
                generator=torch.Generator("cuda").manual_seed(BLEND_SEED)).images[0]
soft = flower_mask.filter(ImageFilter.GaussianBlur(4)).point(lambda v: int(v * KEEP))
backdrop = Image.composite(collage, painted, soft)
backdrop.save("backdrop.png")
side = Image.new("RGB", (2048, 1024)); side.paste(collage, (0, 0)); side.paste(backdrop, (1024, 0))
side.resize((1200, 600))

In [ ]:
files.download("backdrop.png")